# Week 9 skills notebook: probability for people who will be wrong

This notebook belongs to the skills track. It uses dice, a made-up delivery van, a made-up factory and a made-up online shop, so that the skills stay separate from this week's project.

You will measure a probability as a long-run share, use the three rules that every probability obeys, catch an AI assistant that adds chances it should not add, and compute a conditional probability in both directions.

**How to use it:** run each cell in order with the play button or Shift+Enter. Read the output before you move on. There are four "your turn" tasks, and each one has a CHECK cell after it.

## 1. A probability is a long-run share

Roll a fair die many times and count the sixes. The share of sixes jumps around at first and then settles. A probability is the share that an event takes in the long run, so it is always a number from 0 to 1.

The number in `default_rng(9)` is a seed. It makes the random numbers the same every time, so your results match everyone else's.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(9)
rolls = rng.integers(1, 7, size=100_000)       # 100,000 rolls of a fair die
for n in [10, 100, 1_000, 10_000, 100_000]:
    share = (rolls[:n] == 6).mean()
    print(f"share of sixes in the first {n} rolls: {share:.4f}")
print("one in six:", round(1 / 6, 4))

## 2. The complement rule and the sum rule

Two dice have 36 outcomes, and each is equally likely. The cell below lists all 36 in a table. The chance of an event is then the share of rows in which it happens.

- **The complement rule says that the chance of "not A" is 1 minus the chance of A.**
- **The sum rule says that, for two events that cannot happen together, the chance of one or the other is the sum of their chances.** A total of 7 and a double cannot happen on the same roll, so their chances add.

In [ ]:
import itertools

dice = pd.DataFrame(list(itertools.product(range(1, 7), repeat=2)), columns=["first", "second"])
dice["total"] = dice["first"] + dice["second"]
print("rows:", len(dice))

seven = dice["total"] == 7
double = dice["first"] == dice["second"]
print("P(total is 7):", round(seven.mean(), 4))
print("P(total is not 7):", round((~seven).mean(), 4))
print("1 - P(total is 7):", round(1 - seven.mean(), 4))
print("P(double):", round(double.mean(), 4))
print("P(7 or double), counted:", round((seven | double).mean(), 4))
print("P(7) + P(double):", round(seven.mean() + double.mean(), 4))

When two events can happen together, adding their chances counts the overlap twice. "The first die shows 6" and "the second die shows 6" both happen on a double six. The full sum rule subtracts the overlap: P(A or B) = P(A) + P(B) - P(A and B).

In [ ]:
six_first = dice["first"] == 6
six_second = dice["second"] == 6
print("P(first is 6) + P(second is 6):", round(six_first.mean() + six_second.mean(), 4))
print("P(at least one 6), counted:", round((six_first | six_second).mean(), 4))
print("P(both are 6), the overlap:", round((six_first & six_second).mean(), 4))
print("sum minus overlap:", round(six_first.mean() + six_second.mean() - (six_first & six_second).mean(), 4))

## 3. The product rule and independence

Two events are independent when knowing that one happened does not change the chance of the other. For independent events, the chance that both happen is the product of their chances.

The two dice are independent. The first die and the total are not, because a 6 on the first die makes a high total more likely. The cell checks both pairs.

In [ ]:
high = dice["total"] >= 10
print("P(first is 6) x P(second is 6):", round(six_first.mean() * six_second.mean(), 4))
print("P(both are 6), counted:", round((six_first & six_second).mean(), 4))
print()
print("P(first is 6) x P(total >= 10):", round(six_first.mean() * high.mean(), 4))
print("P(first is 6 and total >= 10), counted:", round((six_first & high).mean(), 4))
print("P(total >= 10), all rolls:", round(high.mean(), 4))
print("P(total >= 10 given first is 6):", round(high[six_first].mean(), 4))

The last line is a conditional probability. "Given the first die is 6" means you keep only the rows where the first die is 6 and take the share inside them. Section 8 returns to this idea.

## 4. Beliefs that break the arithmetic

A stated probability is a belief written as a number, and the rules apply to beliefs too. A set of beliefs that breaks a rule is called incoherent. Someone who holds incoherent beliefs can be shown to lose money on bets they themselves call fair.

Three planners each state five chances about two events. A is "the shipment arrives late" and B is "the road to the port is closed". The function below tests four rules and lists the ones a set breaks.

In [ ]:
def check_beliefs(b):
    """b holds P(A), P(not A), P(B), P(A and B) and P(A or B). Returns the rules the set breaks."""
    broken = []
    if any(not 0 <= v <= 1 for v in b.values()):
        broken.append("every chance lies between 0 and 1")
    if abs(b["A"] + b["not A"] - 1) > 1e-9:
        broken.append("P(A) + P(not A) = 1")
    if b["A and B"] > min(b["A"], b["B"]):
        broken.append("P(A and B) is no larger than P(A) or P(B)")
    if abs(b["A or B"] - (b["A"] + b["B"] - b["A and B"])) > 1e-9:
        broken.append("P(A or B) = P(A) + P(B) - P(A and B)")
    return broken

planners = {
    "planner 1": {"A": 0.30, "not A": 0.70, "B": 0.20, "A and B": 0.10, "A or B": 0.40},
    "planner 2": {"A": 0.30, "not A": 0.60, "B": 0.20, "A and B": 0.10, "A or B": 0.40},
    "planner 3": {"A": 0.30, "not A": 0.70, "B": 0.20, "A and B": 0.25, "A or B": 0.25},
}
for name, b in planners.items():
    print(name, b)
    print("    breaks:", check_beliefs(b) or "nothing, the set is coherent")

Planner 3 shows why the rules must be checked together. That planner's "A or B" fits the sum rule exactly, and the set is still incoherent, because two events cannot happen together more often than one of them happens at all.

## 5. Your turn: repair a fourth planner

A fourth planner states the chances below. Run the cell to see which rule the set breaks. Then, in the your-turn cell, store in `fixed_or` the value of P(A or B) that makes the set coherent while the other four chances stay as they are.

In [ ]:
planner_4 = {"A": 0.50, "not A": 0.50, "B": 0.40, "A and B": 0.30, "A or B": 0.70}
print("planner 4 breaks:", check_beliefs(planner_4))

In [ ]:
# your turn 1: store in fixed_or the P(A or B) that makes planner 4 coherent
fixed_or = None

In [ ]:
# CHECK 1
assert fixed_or is not None, "fixed_or is still empty"
assert check_beliefs({**planner_4, "A or B": fixed_or}) == []
print("CHECK 1 passed: planner 4 is coherent with P(A or B) =", round(fixed_or, 2))

## 6. At least one: the AI adds what it should not add

A delivery van breaks down on 20 percent of days, and one day tells you nothing about the next. We asked an AI assistant: *"What is the chance that the van breaks down at least once in a seven-day week?"* It wrote the cell below. Run it, then decide whether the answer is right.

In [ ]:
# Written by an AI assistant
p_day = 0.2
p_week = 7 * p_day
print("Chance of at least one breakdown in a week:", round(p_week, 2))

A chance above 1 is impossible, so the answer is wrong. The assistant added the seven daily chances, and the sum rule allows that only for events that cannot happen together. A breakdown on Monday and a breakdown on Tuesday can both happen.

The safe route goes through the complement. The van gets through the week without a breakdown only if it gets through every one of the seven days. For independent days, the product rule multiplies those seven chances. The chance of at least one breakdown is 1 minus that product. A simulation of 100,000 weeks checks the formula.

In [ ]:
p_no_breakdown_week = (1 - p_day) ** 7
p_at_least_one = 1 - p_no_breakdown_week
print("P(no breakdown all week):", round(p_no_breakdown_week, 4))
print("P(at least one breakdown):", round(p_at_least_one, 4))

rng = np.random.default_rng(19)
weeks = rng.random((100_000, 7)) < p_day        # one row per week, one column per day
print("simulated share of weeks with a breakdown:", round(weeks.any(axis=1).mean(), 4))
print("the AI's answer was too high by:", round(p_week - p_at_least_one, 2))
print("in natural frequencies: about", round(100 * p_at_least_one), "weeks out of 100 have a breakdown")

The same mistake over fewer days gives an answer that looks possible. That case is the dangerous one, because nothing above 1 warns you. The cell compares the added chance with the correct one for one to seven days, and the chart extends the comparison to two weeks.

In [ ]:
import matplotlib.pyplot as plt

for n in [1, 2, 3, 5, 7]:
    print(f"{n} days: added {n * p_day:.2f}, correct {1 - (1 - p_day) ** n:.4f}")

days = np.arange(1, 15)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(days, days * p_day, marker="o", label="added (wrong)")
ax.plot(days, 1 - (1 - p_day) ** days, marker="o", label="1 - (1 - p) ** n")
ax.axhline(1, color="grey", linestyle="--")
ax.set_title("Adding daily chances passes 1, the correct chance never does")
ax.set_xlabel("number of days")
ax.set_ylabel("chance of at least one breakdown")
ax.legend()
plt.tight_layout()
plt.show()

## 7. The fine print of independence

The product rule needs independent days. Many real events come in spells instead. A van that broke down yesterday may still be weak today.

The next cell simulates a second van that breaks down in spells. After a breakdown day, its chance of another breakdown is 50 percent. After a good day, the chance is 12.5 percent. These two numbers keep its long-run share of breakdown days at 20 percent, the same as the first van. Predict before you run it: will this van have more weeks with a breakdown, or fewer?

In [ ]:
rng = np.random.default_rng(29)
n_days = 7 * 100_000                            # 100,000 weeks
u = rng.random(n_days)
broke = np.zeros(n_days, dtype=bool)
broke[0] = u[0] < 0.2
for t in range(1, n_days):
    broke[t] = u[t] < (0.5 if broke[t - 1] else 0.125)

spell_weeks = broke.reshape(-1, 7)              # one row per week
print("share of days with a breakdown:", round(broke.mean(), 4))
print("share of weeks with at least one breakdown:", round(spell_weeks.any(axis=1).mean(), 4))
print("independence formula for the same daily share:", round(1 - (1 - broke.mean()) ** 7, 4))

Both vans break down on about one day in five. The second van has fewer weeks with a breakdown, because its breakdowns bunch into the same weeks and leave other weeks clean. When events cluster, the independence formula overstates the chance of at least one. The fix is to count it: split the record into weeks and take the share of weeks in which the event happened.

## 8. Your turn: a function for "at least one"

Write a function `at_least_one(p, n)` that returns the chance of at least one event in `n` independent tries when each try has chance `p`. You may ask your AI assistant for it. Read its code line by line before you run the CHECK.

In [ ]:
# your turn 2: write the function at_least_one(p, n)
def at_least_one(p, n):
    return None

In [ ]:
# CHECK 2
assert at_least_one(0.2, 7) is not None, "at_least_one still returns nothing"
assert abs(at_least_one(0.2, 7) - (1 - 0.8 ** 7)) < 1e-9
assert abs(at_least_one(0.5, 1) - 0.5) < 1e-9
assert all(0 <= at_least_one(p, n) <= 1 for p in [0.1, 0.5, 0.9] for n in [1, 10, 100])
print("CHECK 2 passed: at_least_one(0.2, 7) =", round(at_least_one(0.2, 7), 4))

## 9. Two questions that sound alike

A made-up factory makes 5,000 items on a day shift and a night shift, and some items are defective. A joint table counts every combination of shift and status, with the totals in the margins. Every conditional probability is a share of one row or one column of this table.

In [ ]:
rng = np.random.default_rng(39)
n = 5_000
shift = np.where(rng.random(n) < 0.3, "night", "day")
broken_item = rng.random(n) < np.where(shift == "night", 0.08, 0.02)
items = pd.DataFrame({"shift": shift, "status": np.where(broken_item, "defective", "fine")})
table = pd.crosstab(items["shift"], items["status"], margins=True)
table

Two questions use the same count and divide it by different totals.

- *Of the night-shift items, what share is defective?* This is P(defective given night). Divide the night-shift defects by the night row's total.
- *Of the defective items, what share came from the night shift?* This is P(night given defective). Divide the same count by the defective column's total.

In [ ]:
night_defects = table.loc["night", "defective"]
p_def_given_night = night_defects / table.loc["night", "All"]
p_night_given_def = night_defects / table.loc["All", "defective"]
print("defective items from the night shift:", night_defects)
print("P(defective given night):", round(p_def_given_night, 3))
print("P(night given defective):", round(p_night_given_def, 3))
print("P(defective given day):", round(table.loc["day", "defective"] / table.loc["day", "All"], 3))

## 10. The assistant answers the other question

A manager asked an AI assistant: *"If an item comes from the night shift, how likely is it to be defective?"* Run its code and compare the answer with the table.

In [ ]:
# Written by an AI assistant
answer = (items.loc[items["status"] == "defective", "shift"] == "night").mean()
print("Chance that a night-shift item is defective:", round(answer, 3))

The assistant kept the defective items and took the share from the night shift. That is P(night given defective), the answer to the other question. The swap has a name, the transposed conditional. In a courtroom it is called the prosecutor's fallacy.

In [ ]:
print("the manager's question, P(defective given night):", round(p_def_given_night, 3))
print("the question the AI answered, P(night given defective):", round(answer, 3))
assert abs(answer - p_night_given_def) < 1e-12
print("the AI's number is P(night given defective)")

## 11. Natural frequencies

Shares of shares are hard to keep straight. A safer format counts items or people out of a round number, such as 1,000. This format is called natural frequencies, and it makes the two questions look as different as they are.

In [ ]:
print("Out of 1,000 night-shift items, about", round(1000 * p_def_given_night), "are defective.")
print("Out of 1,000 defective items, about", round(1000 * p_night_given_def), "come from the night shift.")

The same format exposes the prosecutor's fallacy. A city has 500,000 adults. A witness's description fits one innocent adult in 1,000, and it also fits the person who committed the crime. A suspect matches the description. A prosecutor says: *"Only one innocent person in 1,000 would match, so the chance that this suspect is innocent is one in 1,000."* Count the matches.

In [ ]:
adults = 500_000
p_match_if_innocent = 1 / 1_000
innocent_matches = (adults - 1) * p_match_if_innocent
print(f"adults in the city: {adults:,}")
print("innocent adults who match the description:", round(innocent_matches))
print("guilty adults who match: 1")
p_innocent_given_match = innocent_matches / (innocent_matches + 1)
print("P(match given innocent):", p_match_if_innocent)
print("P(innocent given match):", round(p_innocent_given_match, 4))

The prosecutor quoted P(match given innocent) and called it P(innocent given match). The match alone leaves about 500 innocent adults in the city who fit as well as the suspect does. The match is still evidence, but it needs other evidence beside it before it points at one person.

## 12. Your turn: an online shop's returns

A made-up online shop records how each order was paid, on delivery or by card, and whether the customer returned it. Compute both conditional probabilities from the table:

- `p_return_given_cod`, the share of orders paid on delivery that were returned;
- `p_cod_given_return`, the share of returned orders that were paid on delivery.

In [ ]:
rng = np.random.default_rng(49)
m = 3_000
payment = np.where(rng.random(m) < 0.4, "on delivery", "card")
sent_back = rng.random(m) < np.where(payment == "on delivery", 0.15, 0.05)
orders = pd.DataFrame({"payment": payment, "returned": np.where(sent_back, "returned", "kept")})
shop_table = pd.crosstab(orders["payment"], orders["returned"], margins=True)
shop_table

In [ ]:
# your turn 3: compute both conditional probabilities from the orders table
p_return_given_cod = None
p_cod_given_return = None

In [ ]:
# CHECK 3
assert p_return_given_cod is not None and p_cod_given_return is not None, "a value is still empty"
cod = orders["payment"] == "on delivery"
ret = orders["returned"] == "returned"
assert abs(p_return_given_cod - ret[cod].mean()) < 1e-9, "p_return_given_cod: keep the orders paid on delivery first"
assert abs(p_cod_given_return - cod[ret].mean()) < 1e-9, "p_cod_given_return: keep the returned orders first"
print("CHECK 3 passed:", round(p_return_given_cod, 3), "and", round(p_cod_given_return, 3))

## 13. Your turn: put the pieces together

A driver carries 10 orders paid on delivery to one street. Assume the 10 orders are independent. Use your function from section 8 and your answer from section 12 to store in `p_any_return` the chance that at least one of the 10 orders comes back.

Then write one sentence in natural frequencies that the driver could understand, starting with "Out of 100 such streets".

In [ ]:
# your turn 4: the chance that at least one of 10 orders paid on delivery is returned
p_any_return = None

In [ ]:
# CHECK 4
assert p_any_return is not None, "p_any_return is still empty"
assert abs(p_any_return - (1 - (1 - ret[cod].mean()) ** 10)) < 1e-9
assert p_any_return < 10 * ret[cod].mean()
print("CHECK 4 passed:", round(p_any_return, 3), "against", round(10 * ret[cod].mean(), 3), "if the chances were added")